# Model Training

This notebook trains and compares several models for NBA player points prediction.

The workflow includes:
- loading the processed full-league dataset
- defining input features
- applying a time-based train/test split
- training baseline and machine learning models
- comparing model performance

In [22]:
import pandas as pd
import numpy as np

In [23]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Load processed modeling dataset

This dataset contains:
- player-level historical game data
- rolling features
- trend features
- contextual variables used for prediction

In [24]:
df = pd.read_csv("../data/processed/nba_players_model_dataset.csv")
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
df = df.sort_values("GAME_DATE").reset_index(drop=True)

df.head()

,PLAYER_NAME,PLAYER_ID,GAME_DATE,MATCHUP,PTS,HOME,days_rest,pts_last3,pts_last5,pts_last10,...,fga_last3,fga_last5,fg3a_last5,fta_last5,reb_last5,ast_last5,pts_std_last5,pts_trend,min_trend,fga_trend
0,Cade Cunningham,1630595,2023-11-12,DET @ CHI,10,0,2.0,25.000000,24.6,23.7,...,21.000000,21.8,6.4,5.8,4.2,7.4,5.128353,0.9,0.733333,-0.800000
1,Kevon Looney,1626172,2023-11-12,GSW vs. MIN,2,1,1.0,7.333333,5.6,6.2,...,4.000000,3.2,0.0,1.0,8.2,3.2,4.098780,-0.6,5.333333,0.800000
2,Chris Paul,101108,2023-11-12,GSW vs. MIN,2,1,1.0,11.666667,8.2,8.8,...,8.666667,8.4,3.2,1.0,3.2,6.8,5.932959,-0.6,-0.733333,0.266667
3,Moses Moody,1630541,2023-11-12,GSW vs. MIN,2,1,1.0,7.333333,6.6,7.3,...,5.666667,5.4,3.0,0.8,2.8,1.2,2.966479,-0.7,0.400000,0.266667
4,Marvin Bagley III,1628963,2023-11-12,DET @ CHI,14,0,2.0,9.333333,9.0,10.1,...,6.000000,6.0,0.0,1.8,4.0,1.0,2.828427,-1.1,2.800000,0.000000


## Dataset overview

In [25]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Players:", df["PLAYER_NAME"].nunique())
print("Min date:", df["GAME_DATE"].min().date())
print("Max date:", df["GAME_DATE"].max().date())

Rows: 13201
Columns: 22
Players: 264
Min date: 2023-11-12
Max date: 2024-04-14


## Define input features and target

Target:
- points scored in a game (`PTS`)

Features:
- recent scoring averages
- recent minutes and shot volume
- rolling variability
- trend-based features
- contextual features such as home/away and days of rest

In [26]:
feature_columns = [
    "HOME",
    "days_rest",
    "pts_last3",
    "pts_last5",
    "pts_last10",
    "min_last3",
    "min_last5",
    "fga_last3",
    "fga_last5",
    "fg3a_last5",
    "fta_last5",
    "reb_last5",
    "ast_last5",
    "pts_std_last5",
    "pts_trend",
    "min_trend",
    "fga_trend",
]

target_column = "PTS"

## Time-based train/test split

To simulate a realistic forecasting setup, the dataset is split by date:
- older games are used for training
- newer games are used for testing

In [27]:
split_date = df["GAME_DATE"].quantile(0.8)

train_df = df[df["GAME_DATE"] <= split_date].copy()
test_df = df[df["GAME_DATE"] > split_date].copy()

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

print("Split date:", split_date.date())
print("Train max date:", train_df["GAME_DATE"].max().date())
print("Test min date:", test_df["GAME_DATE"].min().date())
print("Train size:", len(train_df))
print("Test size:", len(test_df))

Split date: 2024-03-18
Train max date: 2024-03-18
Test min date: 2024-03-19
Train size: 10601
Test size: 2600


## Baseline model

As a simple benchmark, I use the player's average points from the last 5 games.

In [28]:
baseline_pred = X_test["pts_last5"]

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("Baseline MAE:", round(baseline_mae, 4))
print("Baseline RMSE:", round(baseline_rmse, 4))
print("Baseline R2:", round(baseline_r2, 4))

Baseline MAE: 5.0955
Baseline RMSE: 6.6798
Baseline R2: 0.4644


## Linear Regression

This model provides a simple interpretable benchmark for tabular prediction.

In [29]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)

## Random Forest

Random Forest is used to capture nonlinear relationships between recent player form and future scoring outcomes.

In [30]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest MAE:", round(rf_mae, 4))
print("Random Forest RMSE:", round(rf_rmse, 4))
print("Random Forest R2:", round(rf_r2, 4))

Random Forest MAE: 4.9081
Random Forest RMSE: 6.4126
Random Forest R2: 0.5064


## Compare model performance

In [31]:
results = pd.DataFrame({
    "model": ["baseline_last5", "linear_regression", "random_forest"],
    "mae": [baseline_mae, lr_mae, rf_mae],
    "rmse": [baseline_rmse, lr_rmse, rf_rmse],
    "r2": [baseline_r2, lr_r2, rf_r2]
})

results.sort_values("mae")

,model,mae,rmse,r2
1,linear_regression,4.882219,6.383890,0.510767
2,random_forest,4.908066,6.412636,0.506352
0,baseline_last5,5.095538,6.679765,0.464367


## Preview predictions on the test set

In [32]:
preds_preview = test_df[["PLAYER_NAME", "GAME_DATE", "MATCHUP", "PTS"]].copy()
preds_preview["baseline_pred"] = baseline_pred.values
preds_preview["lr_pred"] = lr_pred
preds_preview["rf_pred"] = rf_pred

preds_preview.head(10)

,PLAYER_NAME,GAME_DATE,MATCHUP,PTS,baseline_pred,lr_pred,rf_pred
10601,Trey Murphy III,2024-03-19,NOP @ BKN,10,16.6,16.666038,15.999917
10602,Maxi Kleber,2024-03-19,DAL @ SAS,5,1.0,2.997898,4.863138
10603,Jaden Hardy,2024-03-19,DAL @ SAS,4,6.0,5.386274,4.944939
10604,Tre Mann,2024-03-19,CHA @ ORL,11,13.4,12.942108,12.175339
10605,Luka Dončić,2024-03-19,DAL @ SAS,18,31.8,31.669825,31.873537
10606,Brandon Ingram,2024-03-19,NOP @ BKN,11,15.8,19.427967,18.060225
10607,Tre Jones,2024-03-19,SAS vs. DAL,22,11.2,11.190452,11.191311
10608,Goga Bitadze,2024-03-19,ORL vs. CHA,6,2.0,1.885475,2.469057
10609,Mike Conley,2024-03-19,MIN vs. DEN,13,14.0,11.824201,12.068118
10610,Dereck Lively II,2024-03-19,DAL @ SAS,12,11.2,8.702497,9.179302


## Save model evaluation outputs

These files are later used in the evaluation notebook and Streamlit application.

In [33]:
results.to_csv("../models/model_results.csv", index=False)

prediction_df = test_df[["PLAYER_NAME", "GAME_DATE", "MATCHUP", "PTS"]].copy()
prediction_df["baseline_pred"] = baseline_pred.values
prediction_df["lr_pred"] = lr_pred
prediction_df["rf_pred"] = rf_pred
prediction_df["baseline_error"] = prediction_df["PTS"] - prediction_df["baseline_pred"]
prediction_df["lr_error"] = prediction_df["PTS"] - prediction_df["lr_pred"]
prediction_df["rf_error"] = prediction_df["PTS"] - prediction_df["rf_pred"]

prediction_df.to_csv("../models/test_predictions.csv", index=False)

print("Saved:")
print("- ../models/model_results.csv")
print("- ../models/test_predictions.csv")

Saved:
- ../models/model_results.csv
- ../models/test_predictions.csv


## Key observations

At this stage:
- the machine learning models outperform the baseline moderately
- the gain over a simple rolling average is real but not very large
- more contextual features will likely be needed to improve prediction quality further